<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/22-feature-engineering.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 22 — Feature Engineering That Survives

Companion to [the chapter](https://www.ai.biz/books/python-primer/feature-engineering/).


In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(11)


In [ ]:
n = 3000
df = pd.DataFrame({
    'customer_id': rng.integers(1, 400, n),
    'date': pd.Timestamp('2026-01-01') + pd.to_timedelta(rng.integers(0, 180, n), 'D'),
    'revenue': rng.lognormal(4, 0.8, n).round(2),
    'tier': rng.choice(['gold','silver','bronze'], n, p=[.2,.3,.5]),
    'channel': rng.choice(['web','app','store'], n),
}).sort_values(['customer_id','date']).reset_index(drop=True)
df.head()


## 1. Group-relative features — the highest-return family


In [ ]:
df['tier_avg'] = df.groupby('tier').revenue.transform('mean')
df['vs_tier'] = df.revenue - df.tier_avg
df['vs_tier_ratio'] = (df.revenue / df.tier_avg).round(3)
df['rank_in_tier'] = df.groupby('tier').revenue.rank(pct=True).round(3)

print(df[['tier','revenue','tier_avg','vs_tier_ratio','rank_in_tier']].head(8).round(2))
print()
print('£500 means nothing alone. Twice your segment average means a lot.')


## 2. Shift before rolling, or you leak


In [ ]:
g = df.groupby('customer_id').revenue

df['correct_7'] = g.transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
df['LEAKY_7']   = g.transform(lambda s: s.rolling(7, min_periods=1).mean())

one = df[df.customer_id == df.customer_id.iloc[0]].head(5)
print(one[['revenue','correct_7','LEAKY_7']].round(2))
print()
corr_ok = df[['revenue','correct_7']].corr().iloc[0,1]
corr_bad = df[['revenue','LEAKY_7']].corr().iloc[0,1]
print(f'correlation with target — correct: {corr_ok:.3f}, leaky: {corr_bad:.3f}')
print('The leaky window contains the value you are predicting.')


## 3. Fix the snapshot date explicitly


In [ ]:
SNAPSHOT = pd.Timestamp('2026-07-01')      # not Timestamp.now()
agg = df.groupby('customer_id').agg(
    n_orders=('revenue','count'),
    total_revenue=('revenue','sum'),
    avg_revenue=('revenue','mean'),
    first_order=('date','min'),
    last_order=('date','max'),
    n_channels=('channel','nunique'),
)
agg['days_since_last'] = (SNAPSHOT - agg.last_order).dt.days
agg['days_active'] = (agg.last_order - agg.first_order).dt.days
agg['orders_per_month'] = (agg.n_orders / (agg.days_active/30).clip(lower=1)).round(2)
print(agg.head().round(2))
print()
print('Using now() makes features change every run and leaks into backtests.')


## 4. Missingness as signal


In [ ]:
d = pd.DataFrame({'income': rng.lognormal(10, 0.5, 1000)})
# people who decline to state income are different
hide = rng.random(1000) < 0.2
d.loc[hide, 'income'] = np.nan
d['churned'] = ((hide * 1.2) + rng.normal(0, 1, 1000) > 0.8).astype(int)

d['income_missing'] = d.income.isna().astype(int)
print('churn rate when income present:', d.loc[~hide,'churned'].mean().round(3))
print('churn rate when income missing:', d.loc[hide,'churned'].mean().round(3))
print()
print('Impute the gap, but KEEP the flag.')


## 5. Log transform helps linear models, not trees


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

skewed = rng.lognormal(3, 1.5, 2000).reshape(-1,1)
target = (np.log(skewed.ravel()) + rng.normal(0,1,2000) > 3).astype(int)
logged = np.log1p(skewed)

for name, Xd in [('raw', skewed), ('log', logged)]:
    lin = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()),
                          Xd, target, cv=5, scoring='roc_auc').mean()
    tree = cross_val_score(RandomForestClassifier(n_estimators=50, random_state=0),
                           Xd, target, cv=5, scoring='roc_auc').mean()
    print(f'{name}: linear {lin:.3f} | tree {tree:.3f}')
print()
print('Trees split on order, so a monotone transform changes nothing for them.')


## 6. Permutation importance beats built-in importance


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

F = pd.DataFrame({
    'signal': rng.normal(size=2000),
    'noise_binary': rng.integers(0, 2, 2000),
    'noise_highcard': rng.integers(0, 500, 2000),   # many unique values
})
t = (F.signal + rng.normal(0, 0.5, 2000) > 0).astype(int)
Xtr, Xte, ytr, yte = train_test_split(F, t, test_size=0.3, random_state=0)

m = RandomForestClassifier(n_estimators=100, random_state=0).fit(Xtr, ytr)
built_in = pd.Series(m.feature_importances_, index=F.columns)
perm = permutation_importance(m, Xte, yte, n_repeats=10, random_state=0)

print(pd.DataFrame({'built_in': built_in.round(3),
                    'permutation': pd.Series(perm.importances_mean, index=F.columns).round(3)}))
print()
print('Built-in importance inflates the high-cardinality noise column.')


## 7. The production question, as a checklist


In [ ]:
CHECKS = ['When is this computed, relative to the prediction point?',
          'Does it come from a table the outcome updates?',
          'Is it available with the same latency in production?',
          'Would it exist for a brand new customer?']
for i, c in enumerate(CHECKS, 1): print(f'{i}. {c}')
print()
print('Ten minutes with this list catches most leakage.')


## Try it yourself

1. Build a feature from `df.groupby('customer_id').revenue.transform('max')` and decide whether it leaks.
2. Replace the fixed snapshot with `Timestamp.now()` and run twice. Compare.
3. Add `rank_in_channel` and see whether it helps a model more than raw revenue.
